# Day 14 — Random Forest and XGBoost vs. the baseline

Same data, same split, same metrics as the logistic regression baseline. Validation set only; the test set stays untouched until the final comparison.

In [10]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# TODO 1: lengkapi nama module/package.
# RandomForestClassifier ada di module sklearn.xxxx, XGBClassifier ada di package xgboost
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from idxlab.clean import clean_ticker_data
from idxlab.config import load_config
from idxlab.data import DataLoader
from idxlab.evaluation import evaluate_classifier
from idxlab.logging_setup import setup_logging
from idxlab.split import build_dataset, time_based_split

In [2]:
cfg = load_config()
setup_logging(cfg.log_level)

raw = DataLoader.from_config(cfg).load_all()
bbca = clean_ticker_data(raw["BBCA.JK"])

dataset = build_dataset(bbca, horizon=1)
splits = time_based_split(dataset, train_size=0.7, val_size=0.15)

LABEL_COLUMNS = ["forward_return", "direction"]
feature_cols = [c for c in dataset if c not in LABEL_COLUMNS]

assert len(feature_cols) == 12
assert not set(LABEL_COLUMNS) & set(feature_cols), "label bocor ke fitur!"

X_train, y_train = splits.train[feature_cols], splits.train["direction"].astype(int)
X_val, y_val = splits.val[feature_cols], splits.val["direction"].astype(int)

print(f"train {len(X_train)} rows | val {len(X_val)} rows")

10:49:51 | INFO     | idxlab.data | Cache hit for BBCA.JK (data\raw\BBCA_JK.parquet)
10:49:52 | INFO     | idxlab.data | Cache hit for BBRI.JK (data\raw\BBRI_JK.parquet)
10:49:52 | INFO     | idxlab.data | Cache hit for TLKM.JK (data\raw\TLKM_JK.parquet)
10:49:52 | INFO     | idxlab.data | Cache hit for ASII.JK (data\raw\ASII_JK.parquet)
10:49:52 | INFO     | idxlab.data | Cache hit for UNVR.JK (data\raw\UNVR_JK.parquet)


train 595 rows | val 127 rows


In [11]:
models = {
    "dummy": DummyClassifier(strategy="most_frequent"),
    "logreg": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=10,       # TODO 2: parameter batas minimal jumlah sampel di tiap daun
        random_state=42,
        n_jobs=-1,
    ),
    "xgboost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,     # TODO 3: parameter laju pembelajaran tiap pohon
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=300, 
        max_depth=3, 
        num_leaves=8, 
        learning_rate=0.05,
        subsample=0.8, 
        subsample_freq=1, 
        colsample_bytree=0.8,
        min_child_samples=20, 
        random_state=42, 
        verbose=-1,
    ),
}

In [12]:
rows = {}

for name, model in models.items():
    model.fit(X_train, y_train)

    for split_name, X, y in [("train", X_train, y_train), ("val", X_val, y_val)]:
        pred = model.predict(X)
        proba = model.predict_proba(X)[:, 1]
        rows[(name, split_name)] = evaluate_classifier(y, pred, proba)

results = pd.DataFrame.from_dict(rows, orient="index").round(3)    
results    

accuracy  precision  recall     f1  roc_auc
dummy         train     0.565      0.000   0.000  0.000    0.500
              val       0.598      0.000   0.000  0.000    0.500
logreg        train     0.605      0.581   0.332  0.423    0.607
              val       0.583      0.458   0.216  0.293    0.571
random_forest train     0.723      0.805   0.479  0.600    0.819
              val       0.449      0.404   0.784  0.533    0.547
xgboost       train     0.918      0.938   0.869  0.902    0.977
              val       0.480      0.414   0.706  0.522    0.570
lightgbm      train     0.864      0.894   0.780  0.833    0.960
              val       0.496      0.427   0.745  0.543    0.577

In [13]:
comparison = results.xs("val", level=1).sort_values("roc_auc", ascending=False)
comparison

,accuracy,precision,recall,f1,roc_auc
lightgbm,0.496,0.427,0.745,0.543,0.577
logreg,0.583,0.458,0.216,0.293,0.571
xgboost,0.480,0.414,0.706,0.522,0.570
random_forest,0.449,0.404,0.784,0.533,0.547
dummy,0.598,0.000,0.000,0.000,0.500


In [14]:
train_auc = results.xs("train", level=1)["roc_auc"]
val_auc = results.xs("val", level=1)["roc_auc"]

pd.DataFrame({
    "train_auc": train_auc,
    "val_auc": val_auc,
    "gap": train_auc - val_auc,
}).sort_values("gap", ascending=False)

,train_auc,val_auc,gap
xgboost,0.977,0.570,0.407
lightgbm,0.960,0.577,0.383
random_forest,0.819,0.547,0.272
logreg,0.607,0.571,0.036
dummy,0.500,0.500,0.000


In [15]:
for name, model in models.items():
    share_up = model.predict(X_val).mean()
    print(f"{name:14s} predicts 'up' on {share_up:.0%} of validation days")

dummy          predicts 'up' on 0% of validation days
logreg         predicts 'up' on 19% of validation days
random_forest  predicts 'up' on 78% of validation days
xgboost        predicts 'up' on 69% of validation days
lightgbm       predicts 'up' on 70% of validation days


In [16]:
out_dir = Path("docs/results")
out_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(out_dir / "day14_comparison.csv")

## Observations

1. Logreg is the one who's winning on val
2. I thought LGBMClassifier & XGBoost gonna perform great at first, but it's gap is the biggest among the others :)
